# Retrieval quality — evaluare cuantificată

Rulează cele 10 întrebări din `tests.jsonl` prin pipeline-ul de retrieval (`retrieve_with_trace`), calculează MRR / nDCG / recall / precision / hit-rate pe fiecare din cele 4 etape (semantic, lexical, fused, reranked), și le compară.

Presupune că datele sunt deja ingerate în Postgres (vezi `README.md`). Pentru inspecție liberă, nemetrizată, pe întrebări ad-hoc, vezi [`retrieval_playground.ipynb`](../retrieval/retrieval_playground.ipynb).

## Known limitations

- **`reranked` here is `trace.reranked`, not `trace.final`.** `RetrievalTrace.final` is `reranked[:FINAL_K]` — the list the real answer pipeline actually consumes. Today `FINAL_K == RETRIEVAL_K == 20`, so the two lists are identical and every metric below is correct as computed. But this means a `FINAL_K` change alone will **not** show up in this notebook's numbers, since they're computed over the untruncated `reranked` list. If you're specifically evaluating a `FINAL_K` change, add a 5th `evaluate_stage("final", trace.final, test.keywords)` stage before trusting a snapshot diff to reflect it.
- **Keyword matching is over `doc.page_content`.** Every metric here (via `keyword_hit_ranks`, and the reused `calculate_mrr`/`calculate_ndcg` from `eval.py`) checks whether a keyword substring appears in `page_content`. If `ingestion/contextualize.py`'s `add_context` is re-enabled (currently commented out in `ingest.py`), `page_content` becomes LLM-generated context prepended to the real chunk text — a keyword can then match text the LLM wrote *about* the chunk, inflating recall/precision/MRR/nDCG for reasons unrelated to retrieval quality. **A `"contextualize-on"` snapshot is not apples-to-apples with a `"contextualize-off"` one** — don't diff them and conclude contextualize helped without checking by hand. (The breakdown cell below previews `metadata.get("original_text", page_content)`, so once contextualize is on, a "found at [stage]" line can point at text that doesn't itself contain the keyword — the actual match happened in the prepended LLM context.)

In [1]:
from datetime import UTC, datetime

from dotenv import load_dotenv

load_dotenv(override=True)

from app.rag_simple import config
from app.rag_simple.evaluation.retrieval_metrics import (
    average_stage_metrics,
    evaluate_trace,
    keyword_hit_ranks,
)
from app.rag_simple.evaluation.results import EvalSnapshot, diff_snapshots, list_snapshots
from app.rag_simple.evaluation.test import load_tests
from app.rag_simple.retrieval.pipeline import retrieve_with_trace

RESULTS_DIR = config.ROOT / "evaluation" / "results"
STAGES = ("semantic", "lexical", "fused", "reranked")
METRIC_COLUMNS = (
    "mrr",
    "ndcg",
    "recall_at_5",
    "recall_at_10",
    "precision_at_5",
    "precision_at_10",
    "hit_rate_at_10",
)

print(f"RETRIEVAL_K={config.RETRIEVAL_K}  FINAL_K={config.FINAL_K}  CHAT_MODEL={config.CHAT_MODEL}")

/home/alex/projects/RAG/full-stack-fastapi-template/.worktrees/retrieval-quality-eval/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


RETRIEVAL_K=20  FINAL_K=20  CHAT_MODEL=openai:gpt-4.1-mini


In [2]:
tests = load_tests()
traces = []
per_question_metrics = []

for test in tests:
    trace = retrieve_with_trace(test.question)
    traces.append(trace)
    per_question_metrics.append(evaluate_trace(trace, test))

print(f"Evaluated {len(tests)} questions across stages {STAGES}")

Evaluated 10 questions across stages ('semantic', 'lexical', 'fused', 'reranked')


In [3]:
aggregated = average_stage_metrics(per_question_metrics)

header = f"{'stage':<10}" + "".join(f"{col:>16}" for col in METRIC_COLUMNS)
print(header)
print("-" * len(header))
for stage in STAGES:
    row = f"{stage:<10}" + "".join(f"{aggregated[stage][col]:>16.3f}" for col in METRIC_COLUMNS)
    print(row)

stage                  mrr            ndcg     recall_at_5    recall_at_10  precision_at_5 precision_at_10  hit_rate_at_10
--------------------------------------------------------------------------------------------------------------------------
semantic             0.856           0.817           0.920           0.920           0.302           0.193           1.000
lexical              0.554           0.596           0.690           0.742           0.236           0.144           0.900
fused                0.683           0.734           0.860           0.920           0.284           0.175           1.000
reranked             0.899           0.887           0.907           0.920           0.349           0.200           1.000


## Breakdown per întrebare

Pentru fiecare (întrebare, keyword) nefound în etapa finală (reranked), sau al cărui rang se schimbă cu mai mult de `RANK_SHIFT_THRESHOLD` poziții între etape, afișează sursa + heading-ul chunk-ului găsit (la cea mai avansată etapă unde apare) — diagnostic direct al eșecurilor, în stilul `show_chunk` din `retrieval_playground.ipynb`.

In [4]:
RANK_SHIFT_THRESHOLD = 5  # out of RETRIEVAL_K=20, a shift bigger than this is a meaningful stage effect


def show_chunk(doc):
    source = doc.metadata.get("source", "?")
    heading = doc.metadata.get("h4") or doc.metadata.get("h3") or ""
    text = doc.metadata.get("original_text", doc.page_content)
    preview = text.strip().replace("\n", " ")
    if len(preview) > 220:
        preview = preview[:220] + "..."
    return f"{source} | {heading}\n       {preview}"


for test, trace in zip(tests, traces, strict=True):
    stage_docs = {
        "semantic": trace.semantic_hits,
        "lexical": trace.lexical_hits,
        "fused": trace.fused,
        "reranked": trace.reranked,
    }
    for keyword in test.keywords:
        ranks = {stage: keyword_hit_ranks(keyword, docs) for stage, docs in stage_docs.items()}
        found_ranks = [rank for rank in ranks.values() if rank is not None]
        missed_final = ranks["reranked"] is None
        shifted = bool(found_ranks) and (max(found_ranks) - min(found_ranks) > RANK_SHIFT_THRESHOLD)
        if not (missed_final or shifted):
            continue

        print(f"Q: {test.question}")
        print(f"   keyword: {keyword!r}  ranks: {ranks}")
        best_stage = next(
            (stage for stage in ("reranked", "fused", "lexical", "semantic") if ranks[stage] is not None),
            None,
        )
        if best_stage is not None:
            doc = stage_docs[best_stage][ranks[best_stage] - 1]
            print(f"   found at [{best_stage}]: {show_chunk(doc)}")
        else:
            print("   not found at any stage")
        print()

Q: Care sunt proporțiile ideale ale dietei macrobiotice conform diagramei prezentate în seminar?
   keyword: 'cereale complete'  ranks: {'semantic': 1, 'lexical': None, 'fused': 9, 'reranked': 1}
   found at [reranked]: seminar macrobiotica.md | Fructele: Moderație și Metabolizare
       (Diagrama proporțiilor alimentare): *   Cereale complete: 50-60% *   Legume: 25-30% *   Leguminoase și Alge: 10-15% *   Supe: 5%   Chestiunea volumului începe prin a asigura 50% cereale și 50% pentru toate celelalte. În ...

Q: Care sunt proporțiile ideale ale dietei macrobiotice conform diagramei prezentate în seminar?
   keyword: '50-60%'  ranks: {'semantic': 1, 'lexical': None, 'fused': 9, 'reranked': 1}
   found at [reranked]: seminar macrobiotica.md | Fructele: Moderație și Metabolizare
       (Diagrama proporțiilor alimentare): *   Cereale complete: 50-60% *   Legume: 25-30% *   Leguminoase și Alge: 10-15% *   Supe: 5%   Chestiunea volumului începe prin a asigura 50% cereale și 50% pentru toate c

## Salvare snapshot

Editează `LABEL` mai jos înainte de a rula celula (ex. `"baseline"`, `"contextualize-on"`).

In [5]:
LABEL = "baseline"

snapshot = EvalSnapshot(
    label=LABEL,
    timestamp=datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ"),
    config={
        "RETRIEVAL_K": config.RETRIEVAL_K,
        "FINAL_K": config.FINAL_K,
        "CHAT_MODEL": config.CHAT_MODEL,
        "EMBEDDING_MODEL": config.EMBEDDING_MODEL,
    },
    stage_metrics=aggregated,
)
saved_path = snapshot.save(RESULTS_DIR)
print(f"Saved snapshot: {saved_path}")

Saved snapshot: /home/alex/projects/RAG/full-stack-fastapi-template/.worktrees/retrieval-quality-eval/backend/app/rag_simple/evaluation/results/20260918T074453Z_baseline.json


## Comparație before/after

Încarcă ultimele două snapshot-uri salvate și afișează delta per etapă/metrică.

In [6]:
snapshots = list_snapshots(RESULTS_DIR)

if len(snapshots) >= 2:
    older, newer = snapshots[-2], snapshots[-1]
    delta = diff_snapshots(older, newer)
    print(f"Comparing {older.label!r} ({older.timestamp}) -> {newer.label!r} ({newer.timestamp})")
    for stage in STAGES:
        print(f"\n[{stage}]")
        for metric in METRIC_COLUMNS:
            value = delta[stage][metric]
            sign = "+" if value >= 0 else ""
            print(f"  {metric:<16} {sign}{value:.3f}")
else:
    print("Need at least two saved snapshots to compare — run this notebook again after a change.")

Need at least two saved snapshots to compare — run this notebook again after a change.
